# Nonskin_Fusion_On_FC (CLIP·DINO, 학습 없음)

rank 위 피부 밖 융합은 Xception만 통과했다. 여기서는 **학습법 A(fc100_ramp2) 위에 피부 밖 융합을 얹으면, rank 위에 얹은 것보다 나은가**를 본다.
fc는 학습 필터 흔적을 지우지만 원본 AUC를 잃고, 피부 밖 점수는 원본 AUC를 조금 올렸다 → 서로 보완될 수 있는지.

- 전체 얼굴 점수: v2 대각선 `aug_pair_rank_fc100_ramp2@bilateral` seed42. 피부 밖 점수·융합 방식은 rank 위 확인과 동일.
- Xception은 제외: fc@bilateral이 epoch 0(= Baseline)이라 Baseline 위 융합 결과와 같다.
- **사전 판정 기준(2026-10-25, 결과 전 확정), 주 변형 face_inner:**
  - 비교 기준 = rank + face_inner 융합(rank 위 판정표가 고른 w).
  - 통과 = 어떤 w에서 fc + face_inner가 원본 AUC를 비교 기준 대비 −0.01 이내로 유지하면서, 4개 필터 중 3개 이상에서 나쁜 쪽 교차 AUC가 비교 기준 대비 +0.02 이상.
  - 두 쪽 모두 cdf_dev oracle w라 공정한 비교이며, 불합격이면 fc + 피부 밖 조합은 접는다.
- 같은 폴더에 rank 위 결과(`nonskin_fusion_on_rank_bilateral_seed42.csv`)가 먼저 있어야 한다. CDF 공식 test 미사용.

## Cell 1 — 설치·코드 갱신



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, shutil, os

PROJECT=Path('/content/instagram_deepfake_robustness')
if PROJECT.exists():
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/uyh1556/instagram-deepfake-robustness.git',str(PROJECT)],check=True)
assert (PROJECT/'scripts/diagnose_nonskin_fusion.py').is_file(),'새 코드를 GitHub에 push하세요.'
subprocess.run([sys.executable,'-m','pip','install','-q','setuptools==80.9.0','wheel'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-build-isolation','-r',str(PROJECT/'requirements-ffpp-c23-smoothing-eval.txt')],check=True)
subprocess.run([sys.executable,'-c','import torch; assert torch.cuda.is_available(),"GPU 런타임 선택"; print(torch.cuda.get_device_name(0))'],check=True)
print('CPU cores:',os.cpu_count(),'| local free GB:',shutil.disk_usage('/content').free//2**30)


## Cell 2 — 설정·경로



In [ ]:
MODELS=['clip_b16','dinov2_b14']   # Xception fc@bilateral = epoch 0 = Baseline (Baseline 위 융합으로 이미 확인)
ROOT=Path('/content/drive/MyDrive/instagram-deepfake-robustness')
ARCHIVE=ROOT/'manifests/beauty_smoothing_v2/cache'
BASELINE=ROOT/'runs/ffpp_c23_4methods_baseline_v1'
OUT=ROOT/'evaluations/nonskin_fusion_precheck'
RUNS_PARENT=ROOT/'runs'
TRAIN_FILTER='bilateral'
FF_ROOT=Path('/content/ffpp_c23_native_roi16_v1')
CDF_ROOT=Path('/content/celebdf_v2_face_crops16_v1')
MEMBERSHIP=next((ROOT/'manifests/ffpp_c23_smoothing_rank_sda_pool_v2/seed42'/m/'pool_v2/membership_labelled.csv' for m in ['clip_b16','dinov2_b14','xception']
                 if (ROOT/'manifests/ffpp_c23_smoothing_rank_sda_pool_v2/seed42'/m/'pool_v2/membership_labelled.csv').is_file()),None)
assert MEMBERSHIP is not None,'pool v2 membership_labelled.csv를 찾지 못했습니다'
for t in ['ffpp_trainval_mask.tar',*[f'cdf_dev_{p}.tar' for p in ['mask','bilateral','weighted_median','fast_global','domain_transform']]]:
    assert (ARCHIVE/t).is_file(),f'캐시 노트북을 먼저 실행하세요: {t}'
for m in MODELS:
    run=RUNS_PARENT/f'ffpp_c23_smoothing_v2_diagonal_{m}'/f'{m}_{TRAIN_FILTER}_aug_pair_rank_fc100_ramp2_ffpp_c23_smoothing_v2_diagonal_v1_seed42'
    assert (run/'completed.json').is_file(),f'완료된 fc run이 없습니다: {run}'
assert (OUT/f'nonskin_fusion_on_rank_{TRAIN_FILTER}_seed42.csv').is_file(),'rank 위 융합 결과가 먼저 필요합니다(비교 기준)'
print('Models:',MODELS,'| fc train filter:',TRAIN_FILTER,'| out:',OUT)


## Cell 3 — 데이터 준비 (FF++ train/val, CDF 비테스트)



In [ ]:
!python -u "{PROJECT}/scripts/prepare_ffpp_c23_training_data.py" --drive-data "{ROOT}/data/ffpp_c23_4methods_roi16_v1" --data-root "{FF_ROOT}" --trainval-only
assert _exit_code==0,'FF++ train/val 준비 실패'

def extract_cdf(name,size):
    if (CDF_ROOT/'manifests'/f'{name.removesuffix(".tar")}.csv').is_file(): return
    source=ROOT/'data/celebdf_v2_face_crops16_v1'/name
    assert source.is_file() and source.stat().st_size==size,f'TAR 확인: {source}'
    assert shutil.disk_usage('/content').free>2*size+2*2**30,'Colab 로컬 공간 부족'
    local=Path('/content')/name; print('Copying:',name,flush=True); shutil.copy2(source,local)
    subprocess.run(['tar','-xf',str(local),'-C','/content'],check=True); local.unlink()
extract_cdf('celebdf_v2_roi16_non_test_real_v1.tar',826142720)
extract_cdf('celebdf_v2_roi16_non_test_fake_v1.tar',5679564800)


## Cell 4 — 실행 (파싱 + CLIP·DINO, 약 30~50분)



In [ ]:
!python -u "{PROJECT}/scripts/diagnose_nonskin_fusion.py" --full-model fc --runs-parent "{RUNS_PARENT}" --train-filter {TRAIN_FILTER} --source-root "{FF_ROOT}" --cdf-root "{CDF_ROOT}" --membership "{MEMBERSHIP}" --baseline-runs-root "{BASELINE}" --archive-dir "{ARCHIVE}" --output-root "{OUT}" --models {' '.join(MODELS)} --train-images 2000 --val-images 1200 --cdf-images 2000 --batch-size 64 --workers 2
assert _exit_code==0,'fc 위 융합 확인 실패'


## Cell 5 — 결과: fc 단독 대비, 그리고 rank + 피부 밖 대비(판정)



In [ ]:
import pandas as pd
filters=['bilateral','weighted_median','fast_global','domain_transform']
fc=pd.read_csv(OUT/f'nonskin_fusion_on_fc_{TRAIN_FILTER}_seed42.csv')
rk=pd.read_csv(OUT/f'nonskin_fusion_on_rank_{TRAIN_FILTER}_seed42.csv')
rv=pd.read_csv(OUT/f'nonskin_fusion_on_rank_{TRAIN_FILTER}_verdict_seed42.csv')
display(pd.read_csv(OUT/f'nonskin_fusion_on_fc_{TRAIN_FILTER}_verdict_seed42.csv').round(3))
rows=[]
for m in MODELS:
    w_ref=rv[(rv.model==m)&(rv.variant=='face_inner')].w.iloc[0]
    w_ref=0. if pd.isna(w_ref) else w_ref
    ref=rk[(rk.model==m)&(rk.variant=='face_inner')&(rk.w==w_ref)].set_index('filter')
    for w,g in fc[(fc.model==m)&(fc.variant=='face_inner')].groupby('w'):
        g=g.set_index('filter')
        clean=g.normal_auc.iloc[0]-ref.normal_auc.iloc[0]
        gain=(g.worse_cross_auc-ref.worse_cross_auc)[filters]
        rows.append(dict(model=m,rank_ref_w=w_ref,fc_w=w,clean_vs_ref=clean,**{f'gain_{f}':gain[f] for f in filters},
                         filters_improved=int(gain.ge(.02).sum()),passed=bool(clean>=-.01 and gain.ge(.02).sum()>=3)))
cmp=pd.DataFrame(rows); display(cmp.round(3))
print('판정(모델별 통과한 w가 하나라도 있나):',cmp.groupby('model').passed.any().to_dict())